In [2]:
import requests
import os
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup as bs
import csv
import time

### Creating coaches.csv

In [3]:
url = 'https://www.pro-football-reference.com/coaches/'
df = pd.read_html(url)[0]

In [4]:
df.tail()

,Rk,Coach,Yrs,From,To,G,W,L,T,W-L%,...,Yr plyf,G plyf,W plyf,L plyf,W-L%.1,AvRk,BstRk,Chmp,SBwl,Conf
526,527,Johnny Murphy,1,1924,1924,4,0,4,0,0.0,...,0.0,0.0,0.0,0.0,NaN,16.0,16,0.0,0.0,0.0
527,528,Al Nesser,1,1926,1926,2,0,1,1,0.0,...,0.0,0.0,0.0,0.0,NaN,16.0,16,0.0,0.0,0.0
528,529,Tam Rose,1,1921,1921,1,0,1,0,0.0,...,0.0,0.0,0.0,0.0,NaN,18.0,18,0.0,0.0,0.0
529,530,Lenny Sachs,1,1926,1926,4,0,4,0,0.0,...,0.0,0.0,0.0,0.0,NaN,21.0,21,0.0,0.0,0.0
530,531,Giff Smith,1,2023,2023,3,0,3,0,0.0,...,NaN,NaN,NaN,NaN,NaN,4.0,4,NaN,NaN,NaN


In [5]:
soup = bs(requests.get(url).content)

In [6]:
coaches = []
seen = set()
for tag in soup.find_all('a'):
    coach = tag['href'].split('/')[-1].split('.')[0]
    if '/coaches/' in tag['href'] and coach not in seen:
        coaches.append(coach)
        seen.add(coach)

In [7]:
def clean_hof(name):
    if name[-1] == '+' : return name[:-1]
    return name

In [8]:
key = pd.DataFrame(coaches,columns=['id']).iloc[:-1,:]
key['name'] = df['Coach'].apply(clean_hof)
key['to'] = df['To']
key.to_csv('input/coaches.csv')

### Creating photos.csv

In [9]:
from PIL import Image
coaches = pd.read_csv('input/coaches.csv',index_col=0)
coaches.head()

,id,name,to
0,ShulDo0,Don Shula,1995
1,HalaGe0,George Halas,1967
2,BeliBi0,Bill Belichick,2023
3,ReidAn0,Andy Reid,2024
4,LandTo0,Tom Landry,1988


In [10]:
def get_photo(coach,fetch=False):
    path = f'processed/img/{coach}.png'
    if not fetch and os.path.exists(path):
        print(f'path: {path} exists!')
        return True
    else :
        base = 'https://www.pro-football-reference.com/coaches/'
        url = f'{base}/{coach}.htm'
        soup = bs(requests.get(url).content)
        time.sleep(5)
        valid = [img['src'] for img in soup.find_all('img') if 'head' in img['src']]
        if(len(valid) == 0): 
            print(f'no image exists for coach: {coach}')
            return False
        img = valid[0]
        with open(path,'wb') as f:
            f.write(img)
        print(f'image scraped, now at path: {path}')
        return True

In [11]:
def get_all_photos(coaches):
    tqdm.pandas()
    coaches['photo'] = coaches['id'].progress_map(get_photo)
    coaches.to_csv('processed/photos.csv')
    return

In [13]:
# get_all_photos(coaches=coaches)

In [14]:
def get_odds_table(year=2024,fetch=False):
    path = f'processed/odds/ou_{year}.csv'
    if not fetch and os.path.exists(path):
        return pd.read_csv(path,index_col=0)
    else:
        url = f'https://www.sportsoddshistory.com/nfl-win/?y={year}&sa=nfl&t=win&o=t'
        odds = pd.read_html(url)[0].sort_values(by=['Win Total','Over Odds'],ascending=[False,True])
        clean = odds[['Team','Win Total','Actual Wins']].reset_index(drop=True)
        clean.columns = ['Team','OU','Wins']
        clean.to_csv(path)
        time.sleep(20)
        return clean

In [15]:
for year in range(1989,2025):
    get_odds_table(year)

### Updating 2025 Coaches

In [17]:
coaches = pd.read_csv('input/coaches.csv',index_col=0)
active = coaches[coaches['to'] == 2024]

In [23]:
coach = active.iloc[0]['id']
url = f'https://www.pro-football-reference.com/coaches/{coach}.htm'
soup = bs(requests.get(url).content)

In [26]:
pd.read_html(url)[0]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                 Year                Age                 Tm   
0                1999                 41                PHI   
1                2000                 42                PHI   
2                2001                 43                PHI   
3                2002                 44                PHI   
4                2003                 45                PHI   
5                2004                 46                PHI   
6                2005                 47                PHI   
7                2006                 48                PHI   
8                2007                 49                PHI   
9                2008                 50                PHI   
10               2009                 51                PHI   
11               2010                 52                PHI   
12               2011                 53                PHI   
13               2012                 54                PHI   
14               2013                 55                KAN   
15               2014                 56                KAN   
16               2015                 57                KAN   
17               2016                 58                KAN   
18               2017                 59                KAN   
19               2018                 60                KAN   
20               2019                 61                KAN   
21               2020                 62                KAN   
22               2021                 63                KAN   
23               2022                 64                KAN   
24               2023                 65                KAN   
25               2024                 66                KAN   
26             26 yrs             26 yrs                NaN   
27             14 yrs             14 yrs                PHI   
28             12 yrs             12 yrs                KAN   

   Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0  \
                   Lg                  G                  W   
0                 NFL                 16                  5   
1                 NFL                 16                 11   
2                 NFL                 16                 11   
3                 NFL                 16                 12   
4                 NFL                 16                 12   
5                 NFL                 16                 13   
6                 NFL                 16                  6   
7                 NFL                 16                 10   
8                 NFL                 16                  8   
9                 NFL                 16                  9   
10                NFL                 16                 11   
11                NFL                 16                 10   
12                NFL                 16                  8   
13                NFL                 16                  4   
14                NFL                 16                 11   
15                NFL                 16                  9   
16                NFL                 16                 11   
17                NFL                 16                 12   
18                NFL                 16                 10   
19                NFL                 16                 12   
20                NFL                 16                 12   
21                NFL                 16                 14   
22                NFL                 17                 12   
23                NFL                 17                 14   
24                NFL                 17                 11   
25                NFL                 17                 15   
26                NaN                420                273   
27                NaN                224                130   
28                NaN                196                143   

   Unnamed: 6_level_0 Unnamed: 7_level_0 Unnamed: 8_level_0  \
                    L           

In [18]:
for coach in active['id']:
    print(coach)
    url = f'https://www.pro-football-reference.com/coaches/{coach}.htm'
    soup = bs(requests.get(url).content)
    print(soup)
    break


,id,name,to
3,ReidAn0,Andy Reid,2024
11,TomlMi0,Mike Tomlin,2024
12,McCaMi0,Mike McCarthy,2024
14,HarbJo0,John Harbaugh,2024
18,PaytSe0,Sean Payton,2024
